In [4]:
# Importamos Pandas para el manejo de DF
import pandas as pd
# Importamos NumPy para el manejo de vectores
import numpy as np
# Importamos NLTK (Natural Language Tool-Kit)
import nltk
# Importamos el tokenizador de sentencias
from nltk.tokenize import PunktSentenceTokenizer
# Importamos el tokenizador de palabras
from nltk.tokenize import word_tokenize
# Importamos WordNet
from nltk.corpus import wordnet as wn
# Importamos el lexicón de opiniones
from nltk.corpus import opinion_lexicon
# Importamos SentiWordNet
from nltk.corpus import sentiwordnet as swn
# Importamos el algoritmo de WSD simple lesk
from pywsd import simple_lesk
# Importamos las métricas de rendimiento
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
# Importamos las palabras vacías
from nltk.corpus import stopwords
# Importamos el clasificador Naive Bayes
from nltk.classify import NaiveBayesClassifier
# Importamos el clasificador de máxima entropía
from nltk.classify import MaxentClassifier
# Importamos el corpus de opiniones de películas
from nltk.corpus import movie_reviews

english_sw = set(stopwords.words("english"))

# Ejercicio 1

Se quiere desarrollar un evaluador de sentimientos basado en el uso de un recursos lingüístico externo, en concreto, las listas de palabras de opinión que tiene NLTK (opinion:lexicon)

In [15]:
opinion1 = "Visceral, stunning and relentless film making. Dicaprio's Herculean, almost purely physical performance" \
            "and Hardy's wide eyed intensity coupled with the almost overwhelming beauty of the landscape - those " \
            "trees, the natural light, the sun peeking through the clouds, rendered the proceedings down to savage" \
            "poetry. A hypnotic, beautiful, exhausting film."

opinion2 = "I saw this film on Friday. For the first 40 minutes involving spoken dialogue they need not have " \
            "bothered. For me the dialogue was totally unintelligible with grunting, southern states drawl, " \
            "and coarse accent that made it impossible to understand what they were saying."

opinion3 = "It was a idiotic film that produces a magnificent fascination."

In [27]:
def tokenizar(opinion,):
    token = word_tokenize(opinion)
    return token

def calcular_puntuacion(tokens):
    puntuacion = 0
    for token in tokens:
        if token in opinion_lexicon.positive():
            puntuacion+=1
        if token in opinion_lexicon.negative():
            puntuacion-=1
    return puntuacion

In [29]:

print(calcular_puntuacion(tokenizar(opinion1)))
print(calcular_puntuacion(tokenizar(opinion2)))
print(calcular_puntuacion(tokenizar(opinion3)))

1
-4
1


# Ejercicio 2

Se pide lo mismo que en el ejercicio anterior, pero en este caso utilizando SentiWordNet (disponible en NLTK). Esta base de datos proporciona valores positivos y negativos para ciertas palabras en un rango entre -1 y 1. Se puede seguir la misma idea de algoritmo que en el caso anterior, pero hay que tener en cuenta que SentiWordNet nos proporciona puntuaciones para los diferentes sentidos que tiene una palabra. Se puede entonces considerar la puntuación de todos los sentidos de la misma palabra, restando a lo positivo la puntuación negativa. Puede ser interesante que la puntuación global se promedie de acuerdo con el número de sentidos.  Utilizar como entrada las mismas opiniones del ejercicio anterior. ¿El resultado es mejor o peor que el conseguido con el algoritmo del ejercicio 1?

In [51]:
def calcular_puntuacion_swn(text):
    tokens= word_tokenize(text)
    total_score=0
    for t in tokens:
        senti_synsets= list(swn.senti_synsets(t))
        word_score=0
        for ss in senti_synsets:
            word_score+= ss.pos_score()
            word_score-=ss.neg_score()
        if len(senti_synsets)>0:
            total_score+= word_score / len(senti_synsets)
            
        
    return total_score 
print(calcular_puntuacion_swn(opinion1))
print(calcular_puntuacion_swn(opinion2))
print(calcular_puntuacion_swn(opinion3))

0.4219623904464331
-0.25598923992673994
0.7648809523809523


In [ ]:
from nltk.corpus import sentiwordnet as swn
from nltk import word_tokenize

def classify_swn(text):
    # Tokenizar el texto en palabras
    words = word_tokenize(text)
    
    total_score = 0
    num_words = 0  # Contador de palabras procesadas

    for word in words:
        # Obtener los synsets de SentiWordNet para la palabra
        senti_synsets = list(swn.senti_synsets(word))
        
        if not senti_synsets:
            continue  # Si no hay synsets, ignorar la palabra

        # Calcular la puntuación de sentimiento promedio de la palabra
        word_score = 0
        for senti_synset in senti_synsets:
            word_score += (senti_synset.pos_score() - senti_synset.neg_score())
        
        # Promediar la puntuación de la palabra
        total_score += word_score / len(senti_synsets)
        num_words += 1

    # Normalizar la puntuación total
    if num_words > 0:
        return total_score / num_words
    else:
        return 0
    